# Colab: Evaluacija modela

Poredjenje performansi **Custom CNN** i **EfficientNet B0** modela:
- ROC krive i AUC-ROC
- Recall, Precision, F1-score
- Matrice konfuzije

**Preduslovi:** Pokrenite `colab_training_cnn.ipynb` i `colab_training_efficientnet.ipynb` prvo!

## 1. Inicijalizacija

In [ ]:
import os, sys, json

if os.path.exists('/content/colab_paths.json'):
    paths = json.load(open('/content/colab_paths.json'))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    paths = json.load(open('/content/drive/MyDrive/melanoma_results/colab_paths.json'))

sys.path.insert(0, paths['repo_dir'])

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation import compute_metrics, compute_metrics_at_best_threshold
from src.visualization import plot_roc_comparison, plot_confusion_matrix

## 2. Ucitavanje rezultata iz treniranja

In [ ]:
results_dir = paths['results_dir']

with open(os.path.join(results_dir, 'cnn_results.pkl'), 'rb') as f:
    cnn_res = pickle.load(f)

with open(os.path.join(results_dir, 'effnet_results.pkl'), 'rb') as f:
    effnet_res = pickle.load(f)

print(f"CNN - Mean AUC: {cnn_res['mean_auc']:.4f} +/- {cnn_res['std_auc']:.4f}")
print(f"EfficientNet - Mean AUC: {effnet_res['mean_auc']:.4f} +/- {effnet_res['std_auc']:.4f}")

## 3. Poredjenje ROC krivih

ROC kriva prikazuje odnos izmedju True Positive Rate (Recall) i False Positive Rate za razlicite pragove klasifikacije. Veci AUC znaci bolji model.

In [ ]:
fig = plot_roc_comparison({
    'Custom CNN': cnn_res,
    'EfficientNet B0': effnet_res,
})
plt.show()

## 4. Detaljne metrike klasifikacije

Izracunavamo metrike za oba modela koristeci:
1. **Fiksni prag** (0.5)
2. **Optimalni prag** - koji maksimizuje Youden's J statistiku (TPR - FPR)

In [ ]:
# Metrike sa fiksnim pragom 0.5
cnn_metrics = compute_metrics(cnn_res['oof_labels'], cnn_res['oof_probs'], threshold=0.5)
effnet_metrics = compute_metrics(effnet_res['oof_labels'], effnet_res['oof_probs'], threshold=0.5)

# Metrike sa optimalnim pragom
cnn_optimal, cnn_thresh = compute_metrics_at_best_threshold(cnn_res['oof_labels'], cnn_res['oof_probs'])
effnet_optimal, effnet_thresh = compute_metrics_at_best_threshold(effnet_res['oof_labels'], effnet_res['oof_probs'])

comparison = pd.DataFrame({
    'Metrika': ['AUC-ROC', 'Accuracy', 'Recall', 'Precision', 'F1-score', 'Prag'],
    'CNN (prag=0.5)': [
        f"{cnn_metrics['auc_roc']:.4f}",
        f"{cnn_metrics['accuracy']:.4f}",
        f"{cnn_metrics['recall']:.4f}",
        f"{cnn_metrics['precision']:.4f}",
        f"{cnn_metrics['f1']:.4f}",
        '0.5000',
    ],
    'CNN (optimalni)': [
        f"{cnn_optimal['auc_roc']:.4f}",
        f"{cnn_optimal['accuracy']:.4f}",
        f"{cnn_optimal['recall']:.4f}",
        f"{cnn_optimal['precision']:.4f}",
        f"{cnn_optimal['f1']:.4f}",
        f"{cnn_thresh:.4f}",
    ],
    'EfficientNet (prag=0.5)': [
        f"{effnet_metrics['auc_roc']:.4f}",
        f"{effnet_metrics['accuracy']:.4f}",
        f"{effnet_metrics['recall']:.4f}",
        f"{effnet_metrics['precision']:.4f}",
        f"{effnet_metrics['f1']:.4f}",
        '0.5000',
    ],
    'EfficientNet (optimalni)': [
        f"{effnet_optimal['auc_roc']:.4f}",
        f"{effnet_optimal['accuracy']:.4f}",
        f"{effnet_optimal['recall']:.4f}",
        f"{effnet_optimal['precision']:.4f}",
        f"{effnet_optimal['f1']:.4f}",
        f"{effnet_thresh:.4f}",
    ],
})
print("Poredjenje modela (Out-of-Fold predikcije):")
comparison

**Napomena o Recall-u:** U medicinskoj dijagnostici, Recall (senzitivnost) je kriticna metrika jer minimizuje lazno negativne rezultate - ne zelimo da propustimo maligne slucajeve.

## 5. Matrice konfuzije

In [ ]:
print("=== Custom CNN ===")
cnn_preds_opt = (cnn_res['oof_probs'] >= cnn_thresh).astype(int)
fig = plot_confusion_matrix(cnn_res['oof_labels'], cnn_preds_opt,
                            title=f'Custom CNN (prag={cnn_thresh:.3f})')
plt.show()

print("=== EfficientNet B0 ===")
effnet_preds_opt = (effnet_res['oof_probs'] >= effnet_thresh).astype(int)
fig = plot_confusion_matrix(effnet_res['oof_labels'], effnet_preds_opt,
                            title=f'EfficientNet B0 (prag={effnet_thresh:.3f})')
plt.show()

## 6. Classification Report

In [ ]:
print("=== Custom CNN (optimalni prag) ===")
print(cnn_optimal['classification_report'])

print("\n=== EfficientNet B0 (optimalni prag) ===")
print(effnet_optimal['classification_report'])

## Zakljucak

Uporedili smo performanse Custom CNN i EfficientNet B0 modela.

Kljucne metrike za medicinsku primenu:
- **AUC-ROC** - primarna metrika, nezavisna od izbora praga
- **Recall (senzitivnost)** - procenat ispravno detektovanih malignih slucajeva
- **Precision** - procenat tacnih medju svim pozitivnim predikcijama

Sledeci korak: pokrenite `colab_fairness.ipynb` za analizu pravicnosti.